In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.distributed as dist
import os
import yaml
import tarfile
import warnings
import sys
import shutil
from pytorch_lightning import Trainer

# ----------------- Environment adaptive import -----------------
try:
    # Attempt to import FT environment (BioNeMo)
    sys.path.append('/workspace/bionemo') 
    from bionemo.model.molecule.megamolbart.megamolbart_model import MegaMolBARTModel
    from megatron.core import parallel_state
    ENV_TYPE = "FT"
    print("FT (BioNeMo) environment detected.")
except ImportError:
    try:
        # Attempt to import Base environment (Nemo_Chem)
        from nemo_chem.models.megamolbart.megamolbart_model import MegaMolBARTModel
        from apex.transformer import parallel_state
        ENV_TYPE = "BASE"
        print("Base (Nemo_Chem) environment detected.")
    except ImportError:
        print("Unrecognized environment, please check dependencies.")
        sys.exit(1)

warnings.filterwarnings("ignore")

# ================= Configuration =================
#MODEL_PATH = '/models/MegaMolBART.nemo'          # Base model
MODEL_PATH = "/workspace/bionemo/models/checkpoints/MegaMolBART_ft_final.nemo" # Fine-tuned model

OUTPUT_FILENAME = 'LNP-447-bionemo-base.npy'
OUTPUT_DIR = '/tabpfn/LNP-文献配方/LipidNemo_embedding'

CSV_PATH = '/tabpfn/LNP-447.CSV'

COMPONENT_CONFIG = {
    'Ionizable lipid(S)':      'Ionizable lipid(M)',
    'Helper lipid(S)':         'Helper lipid(M)',
    'Cholesterol(S)':          'Cholesterol(M)',
    'PEG(S)':                  'PEG(M)',
    'Additional component(S)': 'Additional component(M)'
}

# Environmental variables
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12355' 
if ENV_TYPE == "FT":
    os.environ["EXPERT_PARALLEL"] = "0"
    os.environ["MODEL_PARALLEL"] = "0"
    os.environ["PIPELINE_PARALLEL"] = "0"
    os.environ['RANK'] = '0'
    os.environ['WORLD_SIZE'] = '1'
# =================================================

def setup_env():
    if not dist.is_initialized():
        dist.init_process_group(backend='nccl', rank=0, world_size=1)
    
    # Clean up old processes
    try:
        if hasattr(parallel_state, 'destroy_model_parallel'):
            parallel_state.destroy_model_parallel()
    except: pass

    # Initialize parallel state
    if ENV_TYPE == "FT":
        parallel_state.initialize_model_parallel(
            tensor_model_parallel_size=1, pipeline_model_parallel_size=1,
            expert_model_parallel_size=1, context_parallel_size=1
        )
    else:
        if not parallel_state.model_parallel_is_initialized():
            parallel_state.initialize_model_parallel(tensor_model_parallel_size_=1, pipeline_model_parallel_size_=1)

def load_model(model_path):
    print(f"Loading model: {model_path}")
    temp_dir = '/tmp/nemo_config_fix_split'
    if os.path.exists(temp_dir): shutil.rmtree(temp_dir)
    os.makedirs(temp_dir, exist_ok=True)
    
    with tarfile.open(model_path, 'r:') as tar:
        member = next(m for m in tar.getmembers() if 'model_config.yaml' in m.name)
        tar.extract(member, path=temp_dir)
    
    with open(os.path.join(temp_dir, 'model_config.yaml'), 'r') as f: config = yaml.safe_load(f)
    config['global_batch_size'] = 1
    config['micro_batch_size'] = 1
    
    override_path = '/tmp/megamolbart_override_split.yaml'
    with open(override_path, 'w') as f: yaml.dump(config, f)
        
    trainer = Trainer(accelerator='gpu', devices=[0], logger=False, enable_checkpointing=False)
    
    # Compatibility loading
    if ENV_TYPE == "FT":
        model = MegaMolBARTModel.restore_from(restore_path=model_path, trainer=trainer, strict=False, override_config_path=override_path)
    else:
        model = MegaMolBARTModel.restore_from(restore_path=model_path, trainer=trainer, override_config_path=override_path)
    
    model.eval().cuda()
    return model

def get_embeddings(model, smiles_list):
    """Extract Embedding for a single component"""
    valid_indices, valid_smiles = [], []
    for i, s in enumerate(smiles_list):
        s_str = str(s).strip()
        # Clean invalid data
        is_invalid = pd.isna(s) or s_str == '' or s_str.lower() == 'nan' or '\\' in s_str
        if not is_invalid:
            valid_indices.append(i)
            valid_smiles.append(s_str)
            
    if not valid_smiles: 
        return np.zeros((len(smiles_list), 512))

    tokenizer = model.tokenizer
    token_ids = [tokenizer.text_to_ids(s) for s in valid_smiles]
    max_len = max(len(ids) for ids in token_ids)
    
    input_ids = torch.full((len(valid_smiles), max_len), tokenizer.pad_id, dtype=torch.long, device='cuda')
    attn_mask = torch.zeros((len(valid_smiles), max_len), dtype=torch.long, device='cuda')
    
    for i, ids in enumerate(token_ids):
        input_ids[i, :len(ids)] = torch.tensor(ids, device='cuda')
        attn_mask[i, :len(ids)] = 1
        
    with torch.no_grad():
        encoder_outputs = model.encode(input_ids, attn_mask)
        mask = attn_mask.unsqueeze(-1)
        embeddings = (torch.sum(encoder_outputs * mask, 1) / (torch.sum(mask, 1) + 1e-9)).cpu().numpy()
        
    final_emb = np.zeros((len(smiles_list), embeddings.shape[1]))
    final_emb[valid_indices] = embeddings
    return final_emb

def main():
    print("Starting independent component feature generation...")
    # === Modification start ===
    print(f"Reading data file: {CSV_PATH}")
    if not os.path.exists(CSV_PATH): 
        print(f"Critical error: File not found -> {CSV_PATH}")
        print("Please check if the path is correct, or if the file has been uploaded.")
        sys.exit(1)
    # === Modification end ===
    #if not os.path.exists(CSV_PATH): sys.exit(1)
    df = pd.read_csv(CSV_PATH)
    # Global cleaning
    df.replace(to_replace=[r'^\\$', r'^\s*\\+\s*$'], value=np.nan, regex=True, inplace=True)

    setup_env()
    model = load_model(MODEL_PATH)
    
    emb_list = []
    ratio_list = []
    
    # Process each component in a loop
    for smi_col, ratio_col in COMPONENT_CONFIG.items():
        #print(f"\nProcessing component pair: [{smi_col}] + [{ratio_col}]")
        
        # 1. Extract Embedding (512 dimensions)
        if smi_col not in df.columns:
            print("   Column does not exist, padding with zeros.")
            raw_emb = np.zeros((len(df), 512))
        else:
            raw_emb = get_embeddings(model, df[smi_col].tolist())
        
        # 3. Extract ratio (1 dimension)
        if ratio_col not in df.columns:
            ratios = np.zeros((len(df), 1))
        else:
            ratios = pd.to_numeric(df[ratio_col], errors='coerce').fillna(0).values.reshape(-1, 1)
            ratios = ratios / 100.0 # Normalize
            
        emb_list.append(raw_emb)
        ratio_list.append(ratios)

    # 5. Final horizontal concatenation
    # First concatenate all Embeddings together
    all_embs = np.hstack(emb_list) # (N, 512*5) = (N, 2560)
    # Then concatenate all Ratios together
    all_rats = np.hstack(ratio_list) # (N, 5)

    X_final = np.hstack([all_embs, all_rats])
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    save_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)
    np.save(save_path, X_final)
    
    print("\n========================================")
    print("Generation complete.")
    print(f"Final matrix shape: {X_final.shape}")
    print(f"Saved to: {save_path}")
    print("========================================")

if __name__ == "__main__":
    main()